# 20 · Travel time band against an independent threshold

**Question.** Does Fieldwork's `travel_time_band` widget select the right junctions — and does
it really take the speed from the junctions rather than restating it?

The arithmetic is a threshold test and is not the interesting part. The **design claim** is:
this widget restates neither the speed nor the direction, reading both from the junctions it is
given, so a band cannot disagree with the traversal that produced it.

That claim is testable, and the reference tests it by recomputing the band budget from the speed
carried **on the junctions** rather than from any parameter of its own.

Criteria fixed before running: the band budget within 5 mm of minutes × the junctions' speed,
the same junctions selected, every remaining budget recomputed against the band, the counts
exact, every kept junction stamped with the band, and the band nested rather than a ring.

In [ ]:
# No geometry library is needed: a band is a threshold on distances the junctions already carry.
print("Nothing to load.")

## Parity

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_20_band import run

report = run(Path.cwd() / "fixtures" / "band-soho-001.json")
print("criterion:", report["criterion"])
print()
print("agrees:", report["agrees"])
print(f"  band                  {report['bandMinutes']} min")
print(f"  speed, from the junctions   {report['speedFromTheJunctions']} m/min")
print(f"  direction, from the junctions {report['directionFromTheJunctions']}")
print(f"  band budget, reference      {report['referenceBandBudgetMetres']:.2f} m")
print(f"  band budget, fieldwork      {report['fieldworkBandBudgetMetres']:.2f} m")
print(f"  traversal budget            {report['traversalBudgetMetres']:.2f} m")
print()
print(f"  junctions offered     {report['junctionsOffered']}")
print(f"  junctions in band     {report['junctionsInBand']}")
print(f"  worst remaining-budget error  {report['worstRemainingBudgetErrorMetres']:.3e} m")
print(f"  nested, not a ring    {report['nestedNotARing']}")

## The part most likely to be got wrong

A junction's **remaining budget is recomputed against the band**, not carried over from the
traversal. A junction 300 m along a 375 m band has 75 m left of the band — not the 825 m it had
left of the 1,125 m traversal it came from.

That matters because the corridor widget uses the remaining budget to decide how far into each
street a traveller gets. Carrying the traversal's figure over would draw a five-minute corridor
with fifteen minutes of reach at its edge.

In [ ]:
fixture = json.loads((Path.cwd() / "fixtures" / "band-soho-001.json").read_text())
budget = report["referenceBandBudgetMetres"]
traversal = report["traversalBudgetMetres"]
offered = {j["networkNodeId"]: j for j in fixture["input"]["junctions"]}

print(f"{'junction':<10} {'travelled':>10} {'left of the band':>18} {'left of the traversal':>23}")
for row in sorted(fixture["output"]["inBand"], key=lambda r: -r["remainingM"])[:3]:
    j = offered[row["networkNodeId"]]
    print(f"{row['networkNodeId']:<10} {j['travelMetres']:>10.1f} {row['remainingM']:>18.1f} "
          f"{j['remainingM']:>23.1f}")
for row in sorted(fixture["output"]["inBand"], key=lambda r: r["remainingM"])[:3]:
    j = offered[row["networkNodeId"]]
    print(f"{row['networkNodeId']:<10} {j['travelMetres']:>10.1f} {row['remainingM']:>18.1f} "
          f"{j['remainingM']:>23.1f}")
print()
print(f"The band is {budget:.0f} m and the traversal was {traversal:.0f} m, so every figure in the")
print(f"third column is {traversal - budget:.0f} m larger than the second. Using it would put")
print("fifteen minutes of reach at the edge of a five-minute corridor.")

## Dive deeper · the assumption is the walking speed

75 metres per minute — 4.5 km/h — is the speed the whole chain rests on. It is not a measured
figure. It is an estimate two authors arrived at from photographs of 1854 London: what the
residents wore, what the streets looked like, how crowded Soho was. They deliberately did not
spend longer on it, because the goal was to represent the Snow event at all.

A band budget is `minutes × speed`, so changing the speed by a factor is the same as changing
the budget by that factor. The fixture carries the distance travelled to all 706 junctions, so
the sensitivity can be measured directly.

In [ ]:
junctions = fixture["input"]["junctions"]
minutes = report["bandMinutes"]
baseline = report["speedFromTheJunctions"]

print(f"{'speed m/min':>12} {'km/h':>6} {'budget m':>10} {'junctions in band':>19} {'vs baseline':>12}")
reference_count = sum(1 for j in junctions if j["travelMetres"] <= minutes * baseline)
for speed in (50, 60, 75, 90, 100):
    budget = minutes * speed
    count = sum(1 for j in junctions if j["travelMetres"] <= budget)
    mark = "  <- assumed" if speed == baseline else ""
    print(f"{speed:>12} {speed * 60 / 1000:>6.1f} {budget:>10.0f} {count:>19} "
          f"{count / reference_count:>11.2f}x{mark}")

print()
print("Walking 20% slower keeps about two thirds of the junctions; 20% faster keeps about half")
print("again more; a third slower halves it. The assumption is not a detail, and it was settled")
print("by looking at photographs of 1854 London.")
print()
print("That is not a criticism of the estimate. It is the reason the speed travels on every")
print("junction: so that nothing downstream can quietly assume a different one.")

### Reading the result

`agrees` means an independent threshold, computed from the speed the junctions carry, selected
exactly the junctions Fieldwork selected and agreed on every remaining budget.

It does not mean 75 m/min is right, and it does not mean the band boundary is where the
five-minute line really is — the boundary falls between junctions, which is what
`reachable_corridor` exists to address.